# Cierre técnico de la Fase 0 en Colab

Esta libreta es un orquestador reproducible. Monta Google Drive, obtiene una revisión identificable del repositorio y llama al código de `src/knee`. No contiene lógica de entrenamiento, no abre la prueba reservada y no publica datos privados.

In [ ]:
REPOSITORY_URL = 'https://github.com/AlonsoYess/Knee.git'
REPOSITORY_REF = 'codex/fase-0-base-reproducible'
REPOSITORY_DIR = '/content/Knee'

In [ ]:
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = userdata.get('KNEE_DATA_ROOT')
if not DRIVE_ROOT:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT con la ruta privada de la tesis en Drive.')

In [ ]:
import subprocess
import sys
from pathlib import Path

repo = Path(REPOSITORY_DIR)
if repo.exists():
    raise RuntimeError(
        'Ya existe /content/Knee. Restablece el entorno de ejecución para evitar usar código anterior.'
    )
subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPOSITORY_REF, REPOSITORY_URL, str(repo)],
    check=True,
)
commit = subprocess.check_output(
    ['git', 'rev-parse', 'HEAD'], cwd=repo, text=True
).strip()
working_tree = subprocess.check_output(
    ['git', 'status', '--porcelain'], cwd=repo, text=True
)
if working_tree.strip():
    raise RuntimeError('El clon no está limpio; la ejecución no es reproducible.')
print(f'Revisión preparada: {commit}')

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(repo)],
    check=True,
)

In [ ]:
import os

data_root = Path(DRIVE_ROOT)
required_sources = [
    data_root / 'data' / 'cohort_input.csv',
    data_root / 'data' / 'cohort_manifest.xlsx',
]
missing = [path.name for path in required_sources if not path.is_file()]
if missing:
    raise FileNotFoundError(f'Faltan fuentes privadas requeridas en Drive: {missing}')
os.environ['KNEE_DATA_ROOT'] = str(data_root)
config = repo / 'configs' / 'paths.example.json'
print('Fuentes privadas localizadas; no se mostrarán identificadores individuales.')

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=repo,
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'knee.change_control',
        '--registry', 'configs/governance/methodology_change_log.json',
    ],
    cwd=repo,
    check=True,
)

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'knee.audit', '--config', str(config)],
    cwd=repo,
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'knee.experiment', '--config', str(config)],
    cwd=repo,
    check=True,
)

In [ ]:
import json

audit_path = data_root / 'outputs' / 'auditorias' / 'auditoria_cohorte.json'
record_path = data_root / 'outputs' / 'auditorias' / 'registro_ejecucion.json'
audit = json.loads(audit_path.read_text(encoding='utf-8'))
record = json.loads(record_path.read_text(encoding='utf-8'))
if audit.get('status') != 'ok':
    raise RuntimeError('La auditoría no terminó correctamente.')
if record.get('git_commit') != commit:
    raise RuntimeError('La bitácora no corresponde al commit ejecutado.')
if record.get('source_sha256') != audit.get('source_sha256'):
    raise RuntimeError('Las huellas de la bitácora y de la auditoría no coinciden.')
print('Fase 0 ejecutada correctamente. Evidencias guardadas en Drive.')
print({'git_commit': commit, 'counts': audit['counts']})